# PySpark 1: how Spark runs your code

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-01-how-spark-runs.ipynb)

A Python script computes daily revenue from one CSV file in a few seconds. Next month the input is 2 TB spread
over thousands of files. Spark runs the same calculation on many machines, but only if you understand what it
does with your code: where it runs, when it runs, and what has to move between machines.

This lesson starts a session, looks at partitions, watches lazy evaluation and a failing transformation, traces
one action through jobs, stages and tasks, and reads the plans Spark builds. Everything later in the course
builds on these ideas.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** basic [Python](../python/learning/python-01-values-and-types.ipynb) and [SQL](../sql/sql-01-basics.ipynb). No Spark knowledge is needed.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Decide when Spark is the right tool

Spark is a distributed engine: it splits data into pieces, processes the pieces in parallel on many cores or
machines, and combines the results. PySpark is its Python API. The heavy lifting happens in the JVM (Java
virtual machine); your Python code describes the work.

That design pays off when the data does not fit on one machine, when the same logic must run as both batch and
streaming, or when a lot of work has to run in parallel. It costs startup time, a cluster to operate, and a more
complex failure model.

**Where it matters**
- **Lakehouse pipelines:** reading terabytes of files from object storage and writing Delta or Iceberg tables.
- **Streaming:** the same DataFrame code processes new events every few seconds.
- **Feature engineering:** one definition computed over years of history for model training.

**Watch out for**
- A few gigabytes fit in memory: pandas, Polars or DuckDB will be simpler and faster.
- Pure SQL over warehouse tables is usually better left to the warehouse (or dbt).
- Spark is not a database: it does not serve low-latency lookups.

## Driver, executors and the cluster manager

A Spark application has three parts:

```text
         +------------------+        asks for resources      +-------------------+
         |      driver      | -----------------------------> |  cluster manager  |
         | your code, plan, |                                 | YARN, Kubernetes, |
         | scheduling       | <----- executors registered --- | standalone        |
         +------------------+                                 +-------------------+
            |   tasks    ^ results
            v            |
    +-------------+  +-------------+  +-------------+
    |  executor   |  |  executor   |  |  executor   |   each runs tasks on its cores
    +-------------+  +-------------+  +-------------+   and keeps data in memory or on disk
```

The **driver** runs your Python program, turns DataFrame code into a plan, splits it into tasks and schedules
them. **Executors** are JVM processes on worker machines; each has a number of cores, and each core runs one
task at a time. The **cluster manager** starts executors. In local mode (`master("local[2]")`), the driver
and a single executor share one process on your machine and `[2]` is the number of cores.

**Where it matters**
- **Sizing:** the driver needs memory for the plan and anything you collect; executors need memory for the data.
- **Failures:** a lost executor's tasks are re-run elsewhere; a lost driver ends the application.

**Watch out for**
- Python runs only on the driver and inside Python UDFs. Built-in DataFrame functions run in the JVM, which is why they are fast.
- `collect()` and `toPandas()` bring the whole result to the driver. On real data that is the most common way to run out of memory.

### Example

Ask the running session what it is: where it runs, how many tasks it can run at once, and its application id.

In [ ]:
sc = spark.sparkContext
print("master:              ", sc.master)
print("default parallelism: ", sc.defaultParallelism)
print("application id:      ", sc.applicationId)
print("Spark version:       ", spark.version)

**What to notice:** `local[2]` means the driver and one executor share this process with two cores, so Spark runs at most two
tasks at a time: `defaultParallelism` is 2. On a cluster, `master` would name YARN, Kubernetes or a standalone
master, and parallelism would be the total number of executor cores.

## Configure the session

`SparkSession` is the entry point to DataFrames, SQL, the catalog and streaming. `SparkSession.builder`
collects settings and `getOrCreate()` returns the session already running in this process or starts a new
one. There are two kinds of settings:

- **Static** settings (memory, cores, `spark.sql.warehouse.dir`) are fixed when the session starts.
- **Runtime SQL** settings (`spark.sql.*`, such as `spark.sql.shuffle.partitions`) can be read and changed at
  any time with `spark.conf.get` and `spark.conf.set`.

**Where it matters**
- **Notebooks on Databricks and similar platforms:** `spark` already exists; you only change runtime settings.
- **Jobs:** `spark-submit` passes static settings, and the code calls `getOrCreate()` (see lesson 14).

**Watch out for**
- Calling `getOrCreate()` again with different builder settings silently returns the existing session; the new static settings are ignored.
- A runtime setting applies to queries run after the change, not to results you already computed.

### Example

Read a runtime setting, change it, and read it back. Then look at a few defaults this session did not change.

In [ ]:
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("after set:         ", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "4")

for key in ["spark.sql.adaptive.enabled", "spark.sql.autoBroadcastJoinThreshold", "spark.sql.ansi.enabled"]:
    print(f"{key:<40} {spark.conf.get(key)}")

**What to notice:** The session cell set shuffle partitions to 4 because the data is tiny; the default is 200. Adaptive query
execution and ANSI SQL mode are on by default in Spark 4, and Spark broadcasts join inputs smaller than about
10 MB (10485760 bytes). Later lessons explain each of these.

## Partitions: how data is split

A DataFrame is split into **partitions**: chunks of rows that one task processes. The number of partitions
sets how much work can run in parallel. It comes from the source (roughly one partition per 128 MB of files,
or the number you ask for), and changes when data is shuffled or explicitly repartitioned.

`df.rdd.getNumPartitions()` returns the count, and `F.spark_partition_id()` tags each row with the partition
that holds it.

**Where it matters**
- **Parallelism:** 2 partitions on a 100-core cluster leave 98 cores idle.
- **Skew:** one partition much larger than the rest makes one slow task.

**Watch out for**
- Too many tiny partitions waste time on scheduling; too few large ones run out of memory. Lesson 13 shows how to choose.
- Partition count is not row count: a partition can be empty.

### Example

Create 10 numbers in 3 partitions and see which partition holds each row; then read the orders as Parquet files and count partitions.

In [ ]:
from pyspark.sql import functions as F

numbers = spark.range(10, numPartitions=3)
print("partitions:", numbers.rdd.getNumPartitions())
numbers.withColumn("partition", F.spark_partition_id()).show()

orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.repartition(3).write.mode("overwrite").parquet(orders_path)
orders_from_files = spark.read.parquet(orders_path)
print("Parquet files written:", len([f for f in os.listdir(orders_path) if f.endswith(".parquet")]))
print("partitions when read back:", orders_from_files.rdd.getNumPartitions())

**What to notice:** Spark split the ten ids into three contiguous ranges, one per partition. Writing three partitions produced three
Parquet files. Reading them back gave a partition count based on file sizes and available cores, not on the
number of files: small files are packed together into fewer partitions.

## Transformations, actions and lazy evaluation

**Transformations** (`select`, `where`, `withColumn`, `join`, `groupBy`) return a new DataFrame and only add a
step to a plan; nothing is read or computed. **Actions** (`count`, `show`, `collect`, `take`, `write`) need a
result, so they make Spark optimize the whole plan and run it. This is **lazy evaluation**.

Because nothing runs until an action, Spark can optimize across all your steps at once: two filters become
one, unused columns are never read. The cost is that errors in a transformation only appear when an action
runs, and every action runs the plan again from the source unless you cache an intermediate result.

**Where it matters**
- **Building large pipelines** step by step costs nothing until the final write.
- **Debugging:** an error that appears at `write` may come from a transformation many lines earlier.

**Watch out for**
- Calling `count()` after every step "to check progress" re-runs the whole pipeline each time.
- `cache()` is lazy too: it stores the data the first time an action computes it.

### Example

Build a transformation that will fail on one row: a cast of `"N/A"` to a number. Defining it succeeds; the error
only appears when an action runs.

In [ ]:
raw = spark.createDataFrame([("1",), ("2",), ("N/A",)], "value string")
doubled = raw.withColumn("doubled", F.col("value").cast("int") * 2)  # a transformation: nothing runs
print("defined without error:", doubled.columns)

spark.sparkContext.setLogLevel("OFF")  # the failure below is expected; keep the log quiet
try:
    doubled.collect()  # an action: now Spark runs the plan
except Exception as exc:
    print("failed at the action:", str(exc).splitlines()[0][:100])
finally:
    spark.sparkContext.setLogLevel("ERROR")

**What to notice:** `withColumn` returned instantly and the new column already appears in the schema, because Spark only checked
names and types. The bad value was found when `collect()` ran the cast: `CAST_INVALID_INPUT`. In a long
pipeline, the line in the stack trace is the action, not the transformation that caused the problem.

### Example: every action recomputes

Run two actions on the same DataFrame, then cache it and run them again.

In [ ]:
paid = orders.where(F.col("status") == "paid").withColumn("amount_with_tax", F.col("amount") * F.lit(1.2))
print("rows:", paid.count(), "| total:", paid.agg(F.sum("amount_with_tax")).first()[0])  # two actions, two runs

paid.cache()
print("cached:", paid.is_cached, "| storage level:", paid.storageLevel)
print("rows:", paid.count())  # this action computes and stores the data
paid.show()  # served from the cache
print("still cached after unpersist:", paid.unpersist().is_cached)

**What to notice:** Without the cache, `count()` and `agg()` each read `orders` and applied the filter again. After `cache()`, the
first action stored the four paid orders in memory and `show()` read them from there. Cache only what several
actions reuse, and `unpersist()` it when you are done.

## Jobs, stages and tasks

Each action starts one or more **jobs**. The scheduler cuts a job into **stages** wherever data has to move
between partitions (a **shuffle**), and each stage runs one **task** per partition.

- **Narrow** transformations (`select`, `where`, `withColumn`) work inside each partition, so they stay in the
  same stage.
- **Wide** transformations (`groupBy`, `join`, `distinct`, `orderBy`) need rows with the same key together, so
  they shuffle and start a new stage. A shuffle writes data to disk and sends it over the network, which is why
  it is the most expensive thing a job does.

**Where it matters**
- **Reading the Spark UI:** jobs, stages and tasks are exactly what its tabs list.
- **Estimating cost:** count the shuffles in a pipeline before you run it.

**Watch out for**
- One action can start several jobs, for example when Spark first measures data for adaptive execution.
- A stage finishes only when its slowest task does.

### Example

Tag the work with a job group, run one aggregation, and ask Spark's status tracker which jobs, stages and tasks
it ran.

In [ ]:
tracker = spark.sparkContext.statusTracker()
spark.sparkContext.setJobGroup("lesson-1", "daily order counts")

by_day = (
    spark.range(1000, numPartitions=4)  # 4 partitions -> 4 tasks before the shuffle
    .withColumn("day", F.col("id") % 7)  # narrow: same stage
    .groupBy("day")  # wide: shuffle, new stage
    .count()
)
by_day.collect()

for job_id in sorted(tracker.getJobIdsForGroup("lesson-1")):
    job = tracker.getJobInfo(job_id)
    stages = [tracker.getStageInfo(s) for s in job.stageIds]
    print(f"job {job_id}: " + ", ".join(f"stage {s.stageId} with {s.numTasks} tasks" for s in stages if s))
spark.sparkContext.setJobGroup("", "")

**What to notice:** One `collect()` produced two jobs. With adaptive execution on, Spark first runs the stage before the shuffle as
a job of its own: 4 tasks, one per input partition, each computing `day` and a partial count. It then looks
at the shuffle output and plans the rest. The second job lists that same stage again (already done, so it is
skipped) and a final stage with a single task: the shuffle produced only seven tiny groups, so adaptive
execution merged the 4 shuffle partitions into one.

## Read the plan Spark builds

Before running anything, Spark turns your code into a plan in four steps:

1. **Parsed** logical plan: your code as written.
2. **Analyzed** logical plan: column names and types resolved against the schema.
3. **Optimized** logical plan: the Catalyst optimizer rewrites it (pushes filters down, drops unused columns,
   folds constants).
4. **Physical** plan: concrete operators (scans, exchanges, joins, aggregates), compiled into fast generated
   code by Tungsten.

`df.explain()` prints the physical plan; `explain("extended")` prints all four; `explain("formatted")` prints
the physical plan with details per operator. Read physical plans from the bottom up.

**Where it matters**
- **Checking an optimization:** whether a filter reached the file reader, or a join is broadcast.
- **Counting shuffles:** each `Exchange` in the physical plan is one.

**Watch out for**
- With adaptive execution on, the plan says `AdaptiveSparkPlan isFinalPlan=false`; Spark may still change it while running.
- `explain` shows the plan, not what happened. The Spark UI shows the runtime facts (lesson 12).

### Example

Print all four plans for a small query. Notice how two filters and a calculation are rewritten.

In [ ]:
query = (
    orders
    .where(F.col("amount") > 50)
    .select("order_id", "amount", (F.col("amount") * 2).alias("double_amount"))
    .where(F.col("order_id") > 6)
)
query.explain("extended")

**What to notice:** In the parsed and analyzed plans the two filters are separate, in the order written. In the optimized plan
Catalyst has merged them into one `Filter` with both conditions and pushed it below the projection, so rows are
dropped before `double_amount` is computed. The physical plan is what actually runs.

### Example: filters pushed to the file reader

Read the Parquet copy of `orders` and look at what reaches the file reader.

In [ ]:
from_files = (
    spark.read.parquet(orders_path)
    .where("amount > 100")
    .select("order_id", "amount")
)
print(plan_of(from_files))

**What to notice:** In the `Scan parquet` node, `PushedFilters` contains `GreaterThan(amount,100.00)`: the Parquet reader checks the
condition itself and can skip whole row groups using the min and max values stored in each file. `ReadSchema`
lists only `order_id` and `amount`, so `status`, `order_ts` and `customer_id` are never read from disk.

## Spark Connect and Spark 4

**Spark Connect** separates the client from the driver. Your Python process sends plans over gRPC to a remote
Spark server instead of starting a driver JVM locally. DataFrame and SQL code are unchanged; only the session
creation differs:

```python
spark = SparkSession.builder.remote("sc://spark-connect.example.internal:15002").getOrCreate()
```

Spark 4.0 (May 2025) also turned **ANSI SQL mode** on by default (invalid casts and overflows raise errors
instead of returning null; lesson 3), added the `VARIANT` type for semi-structured data (lesson 8), Python data
sources and UDTFs (lesson 9), and `transformWithStateInPandas` for custom streaming state (lesson 11).

**Where it matters**
- **Shared clusters and IDEs:** many lightweight clients connect to one server; a client crash cannot take the driver down.
- **Upgrades:** a Spark 3 job that relied on silent nulls from bad casts can fail on Spark 4.

**Watch out for**
- Spark Connect has no RDD API, no `sparkContext`, and no JVM internals such as `df._jdf`. Keep pipeline code in the DataFrame API.
- Read the migration guide before moving a cluster to a new major version.

### RDDs, briefly

Before DataFrames, Spark's API was the **RDD** (resilient distributed dataset): a distributed collection of
Python or JVM objects transformed with functions such as `map` and `filter`. Spark cannot see inside those
functions, so it cannot optimize them, and every row crosses between the JVM and Python. Use DataFrames for
data engineering; you will meet RDDs only in old code and in a few low-level APIs such as `df.rdd.getNumPartitions()`.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What runs on the driver and what runs on the executors?

**A.** The driver runs your program, builds and optimizes the plan, schedules tasks and receives anything you collect. Executors run the tasks on their partitions and hold data in memory or on disk.

**Q2.** What decides how many tasks a stage runs?

**A.** The number of partitions it processes: one task per partition.

**Q3.** Why did the bad cast only fail at `collect()`?

**A.** Transformations are lazy. `withColumn` only added a step to the plan; the data was read and cast when the action ran it.

**Q4.** What is the difference between a narrow and a wide transformation?

**A.** A narrow one works within each partition and stays in the same stage. A wide one needs rows with the same key together, so it shuffles data between executors and starts a new stage.

**Q5.** Two actions run on the same DataFrame. How many times is the source read?

**A.** Twice, unless the DataFrame was cached and materialized by the first action.

**Q6.** Where in a physical plan do you see that a filter reached the file reader?

**A.** In the scan node: `PushedFilters` lists predicates the reader applies, and `ReadSchema` lists the columns it reads.

## Key takeaways

- The driver plans and schedules; executors run one task per partition on each core.
- Transformations build a plan; actions run it. Errors appear at the action.
- Narrow transformations stay in a stage; wide ones shuffle and start a new one.
- Every action recomputes from the source unless you cache.
- Read plans bottom up: scans, filters, exchanges, joins and aggregates.

**Next:** Continue with [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb).